# Examen ICCD753 Recuperación de Información

**Estudiante:** Jairo Angulo  
**Corpus asignado:** NFCorpus

## Enlaces del proyecto

- **Repositorio (GitHub):** https://github.com/Andres-JA/examen-nfcorpu
- **Servicio web (Render):** https://examen-nfcorpu.onrender.com

## Uso del servicio web

| Endpoint | Método | Descripción |
| --- | --- | --- |
| `/health` | GET | Comprobación de disponibilidad |
| `/search` | POST | Búsqueda con TF-IDF (`tfidf`) o BM25 (`bm25`) |


Las consultas se envían por **POST** a `https://examen-nfcorpu.onrender.com/search`
con el encabezado `Content-Type: application/json` y la siguiente estructura:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` acepta `bm25` o `tfidf`.

> **Nota:** el servicio se desplegó en el plan gratuito de Render, que suspende la
> aplicación tras un período de inactividad. Por ello, **la primera solicitud puede
> tardar entre 30 y 60 segundos** mientras el servicio se reactiva. Se recomienda
> abrir primero `GET /health` y luego realizar las consultas.

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

In [1]:
from pathlib import Path
import subprocess
import sys
import json
import hashlib
import importlib.metadata
import pandas as pd
from IPython.display import display, Markdown

BASE = Path.cwd()
for repository in ('nfcorpus', 'nfcorpus-qrels'):
    destination = BASE / 'datos' / repository
    required = destination / ('test.tsv' if repository.endswith('-qrels') else 'corpus/corpus-00000-of-00001.parquet')
    if not required.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', f'https://huggingface.co/datasets/BeIR/{repository}', str(destination)], check=True)
    revision = (subprocess.check_output(['git', '-c', f'safe.directory={destination.as_posix()}', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
                if (destination / '.git').exists() else 'copia local: verificar sha256_datos.json')
    print(repository, 'revisión:', revision)

print('Python:', sys.version.split()[0])
print({name: importlib.metadata.version(name) for name in ['numpy', 'scipy', 'pandas', 'pyarrow', 'scikit-learn', 'nltk', 'flask']})
OUT = BASE / 'resultados'
OUT.mkdir(exist_ok=True)
manifest = {str(p.relative_to(BASE)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted((BASE / 'datos').rglob('*')) if p.suffix in ('.parquet', '.tsv')}
(OUT / 'sha256_datos.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Archivos de datos verificados:', len(manifest))

nfcorpus revisión: b5026a0e96e8a7ac4f95f482a596389289d46269
nfcorpus-qrels revisión: a451b3b26d3ae1358f259c1a3a4dd61fcea35a65
Python: 3.13.16
{'numpy': '2.5.3', 'scipy': '1.18.1', 'pandas': '2.2.3', 'pyarrow': '26.0.1', 'scikit-learn': '1.6.1', 'nltk': '3.10.4', 'flask': '3.1.3'}
Archivos de datos verificados: 5


## 1. Preparación e indexación

- Normalización Unicode NFKC y minúsculas.
- Tokenización con `[a-z0-9]+`: separa signos y guiones; conserva números de términos biomédicos.
- Eliminación de la lista inglesa `ENGLISH_STOP_WORDS` de scikit-learn antes del stemming.
- Stemming Snowball en inglés, sin descargar recursos adicionales. El corpus y las consultas están en inglés.
- Se aplica exactamente el mismo preprocesamiento a documentos y consultas.
- Vocabulario e índice invertido con pares `(fila, frecuencia)`; la fila se relaciona con el `_id` original.

La siguiente celda contiene el código del motor reutilizable: la función `preprocess`
aplica la preparación y el constructor de `SearchEngine` carga los documentos,
construye el vocabulario y el índice invertido. La celda posterior ejecuta esas
operaciones y muestra sus resultados.

In [2]:
%%writefile nfcorpus_ir.py
"""Recuperación de documentos completos de BeIR/NFCorpus."""
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
import math
import re
import unicodedata

import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

STEMMER = SnowballStemmer('english')

@lru_cache(maxsize=100000)
def stem(token):
    return STEMMER.stem(token)

def preprocess(text):
    normalized = unicodedata.normalize('NFKC', str(text)).lower()
    tokens = re.findall(r'[a-z0-9]+', normalized)
    return [stem(t) for t in tokens if t not in ENGLISH_STOP_WORDS]

class SearchEngine:
    def __init__(self, data_dir='datos', k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        paths = sorted((Path(data_dir) / 'nfcorpus' / 'corpus').glob('*.parquet'))
        if not paths:
            raise FileNotFoundError('Primero descarga datos/nfcorpus desde Hugging Face.')
        self.documents = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
        self.documents['_id'] = self.documents['_id'].astype(str)
        self.documents = self.documents.sort_values('_id').reset_index(drop=True)
        assert self.documents['_id'].is_unique, 'Identificadores duplicados'
        self.ids = self.documents['_id'].tolist()
        texts = (self.documents['title'].fillna('') + ' ' + self.documents['text'].fillna('')).tolist()
        self.tokens = [preprocess(t) for t in texts]
        self.lengths = np.array([len(t) for t in self.tokens], dtype=float)
        self.avgdl = float(self.lengths.mean())
        self.inverted_index = defaultdict(list)
        for row, tokens in enumerate(self.tokens):
            for term, frequency in Counter(tokens).items():
                self.inverted_index[term].append((row, frequency))
        self.vocabulary = sorted(self.inverted_index)
        self.vectorizer = TfidfVectorizer(
            analyzer=preprocess, lowercase=False, norm='l2',
            smooth_idf=True, sublinear_tf=True, dtype=np.float64,
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(texts)
        self.bm25_idf = {
            term: math.log(1 + (len(self.ids) - len(postings) + 0.5) / (len(postings) + 0.5))
            for term, postings in self.inverted_index.items()
        }

    def search(self, query, model='bm25', top_k=10):
        if model not in ('bm25', 'tfidf'):
            raise ValueError('model debe ser bm25 o tfidf')
        if not isinstance(query, str) or not query.strip():
            raise ValueError('query debe ser un texto no vacío')
        if type(top_k) is not int or not 1 <= top_k <= len(self.ids):
            raise ValueError(f'top_k debe ser un entero entre 1 y {len(self.ids)}')
        scores = np.zeros(len(self.ids), dtype=float)
        if model == 'tfidf':
            q = self.vectorizer.transform([query])
            scores = (self.tfidf_matrix @ q.T).toarray().ravel()
        else:
            # Cada término distinto de la consulta participa una vez (BM25 sin k3).
            for term in set(preprocess(query)):
                postings = self.inverted_index.get(term)
                if not postings:
                    continue
                rows, frequencies = zip(*postings)
                rows = np.array(rows, dtype=int)
                tf = np.array(frequencies, dtype=float)
                denominator = tf + self.k1 * (1 - self.b + self.b * self.lengths[rows] / self.avgdl)
                scores[rows] += self.bm25_idf[term] * tf * (self.k1 + 1) / denominator
        # Desempate determinista por ID; también completa top_k con puntuaciones 0.
        order = np.argsort(-scores, kind='stable')[:top_k]
        results = []
        for rank, row in enumerate(order, 1):
            doc = self.documents.iloc[int(row)]
            results.append(dict(rank=rank, document_id=doc['_id'], score=float(scores[row]),
                                title=str(doc['title']), snippet=str(doc['text'])[:250]))
        return dict(model=model, results=results)

def load_test(data_dir='datos'):
    base = Path(data_dir)
    paths = sorted((base / 'nfcorpus' / 'queries').glob('*.parquet'))
    queries = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
    queries['_id'] = queries['_id'].astype(str)
    assert queries['_id'].is_unique
    qrels_frame = pd.read_csv(base / 'nfcorpus-qrels' / 'test.tsv', sep='\t',
                              dtype={'query-id': str, 'corpus-id': str})
    qrels = defaultdict(dict)
    for qid, docid, score in qrels_frame[['query-id', 'corpus-id', 'score']].itertuples(index=False, name=None):
        qrels[qid][docid] = max(float(score), qrels[qid].get(docid, 0))
    test_queries = queries[queries['_id'].isin(qrels)].sort_values('_id').reset_index(drop=True)
    assert set(test_queries['_id']) == set(qrels), 'Faltan consultas test'
    return test_queries, dict(qrels), qrels_frame

def metrics_at_10(document_ids, judgments):
    relevant = {docid for docid, value in judgments.items() if value > 0}
    hits = sum(docid in relevant for docid in document_ids[:10])
    # Ganancia lineal, compatible con la convención de trec_eval/BEIR.
    dcg = sum(judgments.get(docid, 0) / math.log2(rank + 2)
              for rank, docid in enumerate(document_ids[:10]))
    ideal = sorted((s for s in judgments.values() if s > 0), reverse=True)[:10]
    idcg = sum(s / math.log2(rank + 2) for rank, s in enumerate(ideal))
    return {'Precision@10': hits / 10,
            'Recall@10': hits / len(relevant) if relevant else 0.0,
            'NDCG@10': dcg / idcg if idcg else 0.0}

def evaluate(engine, queries, qrels):
    records, rankings = [], {'tfidf': {}, 'bm25': {}}
    assert all(docid in set(engine.ids) for js in qrels.values() for docid in js)
    for model in rankings:
        for qid, query in queries[['_id', 'text']].itertuples(index=False, name=None):
            response = engine.search(query, model, 10)
            rankings[model][qid] = response['results']
            ids = [r['document_id'] for r in response['results']]
            records.append({'model': model, 'query_id': qid, **metrics_at_10(ids, qrels[qid])})
    details = pd.DataFrame(records)
    summary = details.groupby('model')[['Precision@10', 'Recall@10', 'NDCG@10']].mean()
    return details, summary, rankings


Overwriting nfcorpus_ir.py


In [3]:
from nfcorpus_ir import SearchEngine, load_test, evaluate, metrics_at_10, preprocess

# Carga, preparación e indexación del corpus completo.
engine = SearchEngine(BASE / 'datos')
display(engine.documents[['_id', 'title', 'text']].head(3))
print('Documentos completos:', len(engine.ids))
print('Términos del vocabulario:', len(engine.vocabulary))
print('Longitud media después del preprocesamiento:', round(engine.avgdl, 2))

# Muestra del texto antes y después del preprocesamiento.
example = 'The effects of dietary fats on cancer cells.'
print('Texto original:', example)
print('Tokens normalizados, sin stopwords y con stemming:', preprocess(example))
print('Primeros 20 términos del vocabulario:', engine.vocabulary[:20])

# Postings con identificadores originales y frecuencias del término.
term = preprocess('cancer')[0]
print('Ejemplo del índice invertido:', term,
      [(engine.ids[row], tf) for row, tf in engine.inverted_index[term][:5]])
assert len(engine.ids) == 3633
assert set(engine.vocabulary) == set(engine.vectorizer.vocabulary_)
print('Identificadores únicos:', len(set(engine.ids)) == len(engine.ids))

,_id,title,text
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an..."
1,MED-1000,Neurobehavioral function and low-level exposur...,Background Animal and in vitro studies demonst...
2,MED-1002,"Polybrominated diphenyl ethers (PBDEs), hydrox...",Prenatal exposure to polybrominated diphenyl e...


Documentos completos: 3633
Términos del vocabulario: 18888
Longitud media después del preprocesamiento: 160.02
Texto original: The effects of dietary fats on cancer cells.
Tokens normalizados, sin stopwords y con stemming: ['effect', 'dietari', 'fat', 'cancer', 'cell']
Primeros 20 términos del vocabulario: ['0', '00', '000', '000006073', '00001', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '001', '0011', '0012', '0013', '0014', '0016']
Ejemplo del índice invertido: cancer [('MED-10', 10), ('MED-1028', 3), ('MED-1045', 3), ('MED-1054', 1), ('MED-1055', 1)]
Identificadores únicos: True


## 2. Modelos

**TF-IDF:** frecuencia sublineal `1 + ln(tf)`, IDF suavizada `ln((1+N)/(1+df))+1`
y normalización L2. El producto de los vectores normalizados equivale al coseno.

**BM25:** `k1=1.2`, `b=0.75`, IDF `ln(1+(N-df+0.5)/(df+0.5))`.
Cada término distinto de la consulta suma `IDF * tf*(k1+1)/(tf+k1*(1-b+b*dl/avgdl))`.
Los parámetros se fijan antes de evaluar; no se ajustan sobre test.

Ambos modelos ordenan por puntuación descendente y desempatan por identificador.
Se devuelven exactamente `top_k` documentos distintos (entre 1 y el tamaño del corpus).
Si faltan coincidencias, se completa con puntuación cero. Una consulta formada solo por
stopwords o términos fuera del vocabulario produce documentos con puntuación cero;
esto indica ausencia de coincidencias léxicas.

In [4]:
# Recuperación con los dos modelos sobre el índice ya construido.
print('Matriz TF-IDF (documentos, términos):', engine.tfidf_matrix.shape)
print('Parámetros BM25:', {'k1': engine.k1, 'b': engine.b})
for model in ('tfidf', 'bm25'):
    print(model)
    display(pd.DataFrame(engine.search('diet and cancer', model, 5)['results']))

Matriz TF-IDF (documentos, términos): (3633, 18888)
Parámetros BM25: {'k1': 1.2, 'b': 0.75}
tfidf


,rank,document_id,score,title,snippet
0,1,MED-2437,0.231473,Diet and breast cancer: understanding risks an...,BACKGROUND: Breast cancer is the most commonly...
1,2,MED-4030,0.217985,Diet and prevention of oral cancer: strategies...,BACKGROUND: Oral health care professionals can...
2,3,MED-5355,0.210484,Intake of whole-grain products and risk of pro...,OBJECTIVE: High intake of whole-grain products...
3,4,MED-4391,0.205947,Lifestyle related causes of cancer and chemopr...,Cancer is a leading cause of death worldwide. ...
4,5,MED-3557,0.204029,VEGETARIAN DIETS AND THE INCIDENCE OF CANCER I...,Background Cancer is the second leading cause ...


bm25


,rank,document_id,score,title,snippet
0,1,MED-3557,5.358482,VEGETARIAN DIETS AND THE INCIDENCE OF CANCER I...,Background Cancer is the second leading cause ...
1,2,MED-4452,5.341128,The role of diet and physical activity in brea...,Background: Evidence for the role of diet and ...
2,3,MED-2424,5.306158,Preschool diet and adult risk of breast cancer.,Events before puberty may affect adult risk of...
3,4,MED-2437,5.294434,Diet and breast cancer: understanding risks an...,BACKGROUND: Breast cancer is the most commonly...
4,5,MED-1333,5.282791,Insulin secretion as a determinant of pancreat...,New epidemiology confirms that glucose intoler...


## 3. Evaluación reproducible con test

Se usan **todas las consultas cuyos identificadores aparecen en `test.tsv`**, enlazadas
con el archivo de consultas original; no se selecciona una muestra ni se mezclan train/dev.
El CSV `resultados/consultas_test.csv` identifica cada consulta y su texto.

Precision@10 = relevantes recuperados / 10; Recall@10 = relevantes recuperados / total
de relevantes de esa consulta. Se considera relevante todo juicio con `score > 0`.
NDCG@10 usa las relevancias graduadas originales y ganancia lineal `rel/log2(rank+1)`,
con rango empezando en 1, siguiendo la convención de trec_eval/BEIR.
Los documentos sin juicio reciben relevancia cero. Se divide DCG por el DCG ideal
de los diez mejores juicios y se promedian las métricas por consulta (media macro).

In [5]:
test_queries, qrels, test_frame = load_test(BASE / 'datos')
print('Consultas test:', len(test_queries))
print('Juicios test:', len(test_frame))
print('Distribución de relevancias:', test_frame['score'].value_counts().sort_index().to_dict())
test_queries[['_id', 'text']].to_csv(OUT / 'consultas_test.csv', index=False)
display(test_queries[['_id', 'text']].head(10))
details, summary, rankings = evaluate(engine, test_queries, qrels)
details.to_csv(OUT / 'metricas_por_consulta.csv', index=False)
summary.to_csv(OUT / 'metricas_resumen.csv')
(OUT / 'rankings_test.json').write_text(json.dumps(rankings, ensure_ascii=False, indent=2), encoding='utf-8')
display(summary.style.format('{:.6f}'))
print('Consultas evaluadas por modelo:', details.groupby('model').size().to_dict())

Consultas test: 323
Juicios test: 12334
Distribución de relevancias: {1: 11758, 2: 576}


,_id,text
0,PLAIN-1008,deafness
1,PLAIN-1018,DHA
2,PLAIN-102,Stopping Heart Disease in Childhood
3,PLAIN-1028,dietary scoring
4,PLAIN-1039,domoic acid
5,PLAIN-1050,Dr. Dean Ornish
6,PLAIN-1066,Dr. Walter Willett
7,PLAIN-1088,ECMO
8,PLAIN-1098,eggnog
9,PLAIN-1109,endocrine disruptors


,Precision@10,Recall@10,NDCG@10
model,,,
bm25,0.238390,0.156516,0.328759
tfidf,0.232198,0.156192,0.323836


Consultas evaluadas por modelo: {'bm25': 323, 'tfidf': 323}


In [6]:
# Verificación manual de una consulta y ejemplo con resultado ideal conocido.
assert metrics_at_10(['a', 'b'], {'a': 2, 'b': 1}) == {'Precision@10': 0.2, 'Recall@10': 1.0, 'NDCG@10': 1.0}
qid = test_queries.iloc[0]['_id']
print('Consulta de ejemplo:', qid, test_queries.iloc[0]['text'])
for model in ('tfidf', 'bm25'):
    table = pd.DataFrame(rankings[model][qid])
    table['relevancia_qrel'] = table['document_id'].map(qrels[qid]).fillna(0)
    print(model, metrics_at_10(table['document_id'].tolist(), qrels[qid]))
    display(table[['rank', 'document_id', 'score', 'relevancia_qrel']])

Consulta de ejemplo: PLAIN-1008 deafness
tfidf {'Precision@10': 0.0, 'Recall@10': 0.0, 'NDCG@10': 0.0}


,rank,document_id,score,relevancia_qrel
0,1,MED-4668,0.156704,0.0
1,2,MED-10,0.000000,0.0
2,3,MED-1000,0.000000,0.0
3,4,MED-1002,0.000000,0.0
4,5,MED-1003,0.000000,0.0
5,6,MED-1004,0.000000,0.0
6,7,MED-1005,0.000000,0.0
7,8,MED-1006,0.000000,0.0
8,9,MED-1007,0.000000,0.0
9,10,MED-1008,0.000000,0.0


bm25 {'Precision@10': 0.0, 'Recall@10': 0.0, 'NDCG@10': 0.0}


,rank,document_id,score,relevancia_qrel
0,1,MED-4668,9.261055,0.0
1,2,MED-10,0.000000,0.0
2,3,MED-1000,0.000000,0.0
3,4,MED-1002,0.000000,0.0
4,5,MED-1003,0.000000,0.0
5,6,MED-1004,0.000000,0.0
6,7,MED-1005,0.000000,0.0
7,8,MED-1006,0.000000,0.0
8,9,MED-1007,0.000000,0.0
9,10,MED-1008,0.000000,0.0


In [7]:
delta = summary.loc['bm25'] - summary.loc['tfidf']
display(Markdown('### Comparación e interpretación'))
for metric in summary.columns:
    best = summary[metric].idxmax()
    print(f'{metric}: {best} obtiene el mayor promedio. Diferencia BM25 - TF-IDF = {delta[metric]:+.6f}')
display(Markdown(
    'BM25 supera ligeramente a TF-IDF en las tres métricas en esta ejecución. '
    'BM25 incorpora saturación de frecuencia y normalización de longitud, mientras TF-IDF usa coseno. '
    'La diferencia observada es descriptiva: no se realizó una prueba de significancia. '
    'Precision@10 cercana a 0,24 significa aproximadamente 2,4 documentos juzgados relevantes '
    'entre los diez primeros por consulta. Recall@10 es menor porque algunas consultas tienen '
    'muchos documentos relevantes y solo se recuperan diez. Ambos métodos son léxicos y pueden '
    'omitir sinónimos o relaciones semánticas.'))

### Comparación e interpretación

Precision@10: bm25 obtiene el mayor promedio. Diferencia BM25 - TF-IDF = +0.006192
Recall@10: bm25 obtiene el mayor promedio. Diferencia BM25 - TF-IDF = +0.000324
NDCG@10: bm25 obtiene el mayor promedio. Diferencia BM25 - TF-IDF = +0.004923


BM25 supera ligeramente a TF-IDF en las tres métricas en esta ejecución. BM25 incorpora saturación de frecuencia y normalización de longitud, mientras TF-IDF usa coseno. La diferencia observada es descriptiva: no se realizó una prueba de significancia. Precision@10 cercana a 0,24 significa aproximadamente 2,4 documentos juzgados relevantes entre los diez primeros por consulta. Recall@10 es menor porque algunas consultas tienen muchos documentos relevantes y solo se recuperan diez. Ambos métodos son léxicos y pueden omitir sinónimos o relaciones semánticas.

## 4. Servicio web y pruebas del contrato

`GET /health` confirma disponibilidad. La búsqueda usa exclusivamente `POST /search`,
como pide el enunciado. Recibe un cuerpo JSON con `query`, `model` (`tfidf` o `bm25`)
y `top_k`, y la cabecera `Content-Type: application/json`.
La búsqueda devuelve `model` y `results`, con
`rank`, `document_id`, `score`, `title` y `snippet`. Los qrels no intervienen en la búsqueda.
Los cuerpos inválidos devuelven 400, y el tipo de contenido incorrecto devuelve 415.

In [8]:
%%writefile app.py
"""API del examen: python app.py (PORT configurable)."""
import os
from pathlib import Path
from flask import Flask, jsonify, request
from nfcorpus_ir import SearchEngine

def create_app(engine=None):
    app = Flask(__name__)
    engine = engine if engine is not None else SearchEngine(Path(__file__).parent / 'datos')

    @app.get('/health')
    def health():
        return jsonify(status='ok', corpus='nfcorpus', documents=len(engine.ids), models=['tfidf', 'bm25'])

    @app.post('/search')
    def search():
        if not request.is_json:
            return jsonify(error='Se requiere Content-Type: application/json'), 415
        payload = request.get_json(silent=True)
        if not isinstance(payload, dict):
            return jsonify(error='El cuerpo debe ser un objeto JSON válido'), 400
        try:
            return jsonify(engine.search(payload.get('query'), payload.get('model', 'bm25'), payload.get('top_k', 10)))
        except (ValueError, TypeError) as exc:
            return jsonify(error=str(exc)), 400

    return app

if __name__ == '__main__':
    create_app().run(host='0.0.0.0', port=int(os.environ.get('PORT', 8000)), debug=False)


Overwriting app.py


In [9]:
from app import create_app
client = create_app(engine).test_client()
response = client.get('/health')
assert response.status_code == 200
print('GET /health:', response.get_json())
for model in ('tfidf', 'bm25'):
    # Consulta libre: no depende de conocer su ID ni de consultar los qrels.
    parameters = {'query': 'nutritional effects of coffee consumption', 'model': model, 'top_k': 7}
    response = client.post('/search', json=parameters)
    assert response.status_code == 200
    result = response.get_json()
    rows = result['results']
    assert result['model'] == model and len(rows) == 7
    assert len({r['document_id'] for r in rows}) == 7
    assert all(r['document_id'] in engine.ids for r in rows)
    assert [r['rank'] for r in rows] == list(range(1, 8))
    assert all(rows[i]['score'] >= rows[i+1]['score'] for i in range(len(rows)-1))
    print(model, 'POST HTTP', response.status_code, '— contrato, orden, IDs y top_k correctos')
    print(json.dumps({'model': model, 'results': rows[:1]}, ensure_ascii=False, indent=2))
for payload in ({'query': ''}, {'query': 'coffee', 'model': 'otro'},
                {'query': 'coffee', 'top_k': 0}, {'query': 'coffee', 'top_k': True}):
    assert client.post('/search', json=payload).status_code == 400
assert client.post('/search', data='texto').status_code == 415
assert client.post('/search', data='{', content_type='application/json').status_code == 400
assert client.get('/search').status_code == 405
assert len(client.post('/search', json={'query': 'coffee'}).get_json()['results']) == 10
print('GET /search: HTTP 405, según el contrato de búsqueda por POST.')
for model in ('tfidf', 'bm25'):
    rows = engine.search('zzzzinexistentetermzzz', model, 10)['results']
    assert len(rows) == 10 and all(r['score'] == 0 for r in rows)
print('Pruebas de entradas inválidas y consulta fuera del vocabulario: correctas.')

GET /health: {'corpus': 'nfcorpus', 'documents': 3633, 'models': ['tfidf', 'bm25'], 'status': 'ok'}
tfidf POST HTTP 200 — contrato, orden, IDs y top_k correctos
{
  "model": "tfidf",
  "results": [
    {
      "document_id": "MED-1636",
      "rank": 1,
      "score": 0.3259829574927339,
      "snippet": "Coffee drinking has been associated with increased serum cholesterol levels in some, but not all, studies. A Medline search of the English-language literature published prior to December 1998, a bibliography review, and consultations with experts wer",
      "title": "Coffee consumption and serum lipids: a meta-analysis of randomized controlled clinical trials."
    }
  ]
}
bm25 POST HTTP 200 — contrato, orden, IDs y top_k correctos
{
  "model": "bm25",
  "results": [
    {
      "document_id": "MED-1649",
      "rank": 1,
      "score": 12.675117193863574,
      "snippet": "OBJECTIVE: The association of coffee consumption with cardiovascular disease remains controversial. Endothelial

## 5. URL pública y uso desde Postman

Iniciar la API desde una terminal: `python app.py` (puerto 8000). Para alojamiento
persistente se incluye un `Dockerfile` con Waitress: publicar su puerto 8000 en un
proveedor con contenedores y usar la URL HTTPS que asigne. El corpus completo se incluye
en la imagen; los juicios y las consultas no son necesarios para servir búsquedas.

URL del servicio desplegado por el estudiante: `https://examen-nfcorpu.onrender.com`.
Para actualizar Render, subir el `app.py` actualizado al repositorio y desplegar
su nuevo commit. Las pruebas de esta ejecución verifican el código local actualizado.

En Postman usar `POST https://examen-nfcorpu.onrender.com/search`.
Seleccionar **Body → raw → JSON**, con cabecera `Content-Type: application/json`, y enviar:
```json
{"query": "diet and cancer", "model": "bm25", "top_k": 10}
```
Repetir con `"model": "tfidf"` y consultas propias en inglés.
Para comprobar disponibilidad en el navegador: `GET https://examen-nfcorpu.onrender.com/health`.
Abrir `/search` en el navegador envía GET y produce HTTP 405.

In [10]:
url_file = BASE / 'resultados' / 'url_publica.txt'
if url_file.exists():
    print(url_file.read_text(encoding='utf-8'))
else:
    print('URL: https://examen-nfcorpu.onrender.com — actualizar app.py y redesplegar para aplicar el contrato de búsqueda exclusivamente POST.')
print('Entregable: resolucion_nfcorpus.ipynb con salidas, métricas y código del servicio.')

URL: https://examen-nfcorpu.onrender.com — actualizar app.py y redesplegar para aplicar el contrato de búsqueda exclusivamente POST.
Entregable: resolucion_nfcorpus.ipynb con salidas, métricas y código del servicio.
